## Flash attention

#### What's wrong with normal softmax attention?

- we always need to materialize the full $O(T^2)$ matrix which is very expensive at long context lengths, where $T$ = max_seq_len
- consider the size of the Query matrix
  - sequence length x embedding dimension: $O(Td)$
- now consider the size of the $QK^T$ matrix (i.e. the query-key matrix)
  - sequence length x sequence length $O(T^2)$

For max sequence length $T$ = 128k, embed dim $d$ = 8192

- 128k x 128k is an order of magnitude larger than 128k x 8192

So we really want to shrink how much of this $QK^T$ matrix we materalize! Especially since frontier models now have >1M context length 🤯

#### How do we do this?

So we might deduce that the first row of the attention output is actually just

- First row of the query matrix $Q$, dotted with the first row of the K matrix (i.e. the first query token ($Q_1$), 'attends' to the first key token ($K$))
- Then that output forms the first element of our $QK^T$ matrix!
- But...we can't do softmax with just this one value, we need the max and the exponentiated sum of the entire row

Recall: $ softmax(x_1) = \dfrac{e^{x_1 - max_1}}{\sum_i^T e^{x_i - max_1}}$

<img src="../assets/flash-1.png" width="300">

Source: [Priyam](https://www.youtube.com/watch?v=CwLelq5SX7g)

_Why not just do the first query row dotted with all the columns in $K^T$?_

- This still requires us to materialize K - $O(Td)$ which we cannot load into SRAM (on-chip shared memory)
- We want to fit all our calculations onto a 100-200kb chip (SRAM)

#### Online softmax

Lets find the max and the exponentiated sum (denominator of softmax) for each row - once we have that, we can find each element of the final attention matrix iteratively!

Thus, never needing to materialize the full $QK^T$ or $K, V$ matrices! In fact we'll only ever need a row and column from $Q, K$ to get our softmax output!

Here's the brief formula below where $m_i$ represents the max of row i, and $d_i$ represents the exponentiated sum of row i (running sum):

<img src="../assets/flash-2.png" width="300">

Source: [Priyam](https://www.youtube.com/watch?v=CwLelq5SX7g)


In [ ]:
import torch
import torch.nn as nn 
import math
import time

# input is the QK^T matrix: (T, T)
def softmax(x):

	row_max = torch.max(x, axis=1, keepdims=True).values # max value for every token
	x_stable = x-row_max 

	exp_x = torch.exp(x_stable)
	sum_x = torch.sum(exp_x, dim=1, keepdims=True) # sum along every row, getting the summed logits

	return exp_x/sum_x


# input is (T, d)
def online_softmax(x):

	# the trick of online softmax is that we compute the max and the exponentiated sum for every row
	# this allows us to chunk of the QK^T matrix, never materializing the full O(T^2) tensor

	M, N = x.shape # materialize a part of the T,T matrix 
	
	m_store = torch.zeros((M,)) # store the max for each row
	d_store = torch.zeros((M,)) # store the exponentiated sum for each row

	for i in range(M):

		xi = x[i] # taking the i'th token in x 

		m = float('-inf')
		d = 0 

		for j in range(T):

			xj = xi[j] # taking the j'th token, that token i would be attending to
			
			m_old = m
			m = max(m, xj) # set m to the maximum of these 2 numbers 
			
			# running exponentiated sum
			d = math.exp(xj-m) + d * math.exp(m_old-m) # d_{i+1} = d_i * exp(m_i - m_{t+1}) + exp(x_i - m_{i+1}) - update the sum with the new max, then add the new exponentiated value
		
		m_store[i] = m
		d_store[i] = d

	return m_store, d_store

def online_softmax_with_logsumexp(x):
	
	M, N = x.shape

	# logsumexp trick is basically that softmax(x) = x - (max + log(sum of all exponentiated terms))
	L_store = torch.zeros((M,))

	for i in range(M):

		xi = x[i]
		m = float('-inf')
		d = 0

		for j in range(T):

			xj = xi[j]

			m_old = m
			m = max(xj, m)
			d = d * math.exp(m_old-m) + math.exp(xj-m)

		L_store[i] = m + math.log(d)

	return L_store

def online_softmax_chunked(x, block_size=64):
	
	M, N = x.shape # differs for decode, chunking like in flash attention, cross-attention

	num_blocks = math.ceil(N / block_size)

	L_store = torch.zeros((M,))

	for i in range(M):

		xi = x[i]
		
		m = float('-inf')
		d = 0

		for j in range(num_blocks):

			block_start = j * block_size
			block_end = min(block_start + block_size, N) # clamping
			block = xi[block_start:block_end] # processes the tokens in the key matrix in chunks!

			block_max = block.max() # max of this chunk plz!
			m_old = m 
			m = max(m, block_max) # compare with prev block max

			d = d * math.exp(m_old - m) + torch.exp(block - m).sum().item() # sum of the previous blocks + new sum of this block!

		L_store[i] = m + torch.log(d) # add the block values to the block store! 

x = torch.randn(1024, 1024)
t0 = time.perf_counter()
online_softmax_with_logsumexp(x)
t1 = time.perf_counter()-t0
t1 # woah this is hella slow

1.9293024581857026